<a href="https://colab.research.google.com/github/TrungTan369/Machine-Learning-Assignment/blob/main/notebooks/ex3_imageData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello world from colab web font-end")

Hello world from colab web font-end


In [ ]:
# Cell 1: Tải dataset từ Kaggle (kagglehub)
# Cài đặt và tải dataset INRIA (chạy trên Colab)
%pip install kagglehub -q
import kagglehub, os
print("Downloading INRIA Person from Kaggle...")
dataset_path = kagglehub.dataset_download("jcoral02/inriaperson")
print("Done. dataset_path =", dataset_path)

Bắt đầu tải dataset INRIA Person từ Kaggle...
Using Colab cache for faster access to the 'inriaperson' dataset.
Đã tải xong! Đường dẫn dữ liệu: /kaggle/input/inriaperson


In [ ]:
# Machine Learning Traditional
# ...existing code...
# Cell: Load dataset, EDA cơ bản, feature extraction và lưu đặc trưng
import os
import zipfile
import shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
# dataset_path được tạo bởi cell trước (kagglehub)
print("Raw dataset_path:", dataset_path)

# Nếu là zip thì giải nén
if isinstance(dataset_path, str) and dataset_path.endswith('.zip'):
    extract_dir = '/content/inriaperson'
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(dataset_path, 'r') as z:
        z.extractall(extract_dir)
    dataset_root = extract_dir
else:
    dataset_root = dataset_path

# Tìm thư mục root chứa folder lớp ảnh
def find_image_root(root):
    p = Path(root)
    # nếu p trực tiếp chứa folder lớp -> trả về p
    for child in p.iterdir():
        if child.is_dir() and any(child.glob('*.[jp][pn]g')):
            return str(p)
    # ngược lại tìm thư mục con đầu tiên có ảnh
    for sub in p.rglob('*'):
        if sub.is_dir() and any(sub.glob('*.[jp][pn]g')):
            return str(sub)
    return str(p)

dataset_for_loader = find_image_root(dataset_root)
print("Using dataset root for loader:", dataset_for_loader)

# Cấu hình (chỉnh nếu cần)
image_size = (224, 224)
batch_size = 32
model_name = 'resnet50'  # 'vgg16' hoặc 'resnet50'
pooling = 'avg'

from modules.image_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk

# Load (resize theo image_size)
print("1) Load và tiền xử lý (resize):")
X_raw, y_raw, class_names = load_and_preprocess_data(dataset_for_loader, image_size=image_size, batch_size=batch_size, shuffle=True)
print("X_raw.shape:", X_raw.shape, "y_raw.shape:", y_raw.shape)
print("Classes:", class_names)

# EDA cơ bản
print("\n--- EDA cơ bản ---")
n_samples = X_raw.shape[0]
n_classes = len(class_names)
print(f"Tổng số ảnh: {n_samples}, Số lớp: {n_classes}")

# phân phối nhãn
(unique, counts) = np.unique(y_raw, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u} ({class_names[u]}): {c} samples")

# pixel stats
print("Pixel dtype:", X_raw.dtype, "min/max:", X_raw.min(), X_raw.max(), "mean:", X_raw.mean())

# Hiển thị mẫu (1 ảnh mỗi lớp, tối đa 6 lớp)
to_show = min(n_classes, 6)
fig, axs = plt.subplots(1, to_show, figsize=(3*to_show,3))
for i in range(to_show):
    idx = np.where(y_raw==unique[i])[0][0]
    axs[i].imshow((X_raw[idx].astype(np.uint8)))
    axs[i].set_title(f"{class_names[unique[i]]}\ncount={counts[i]}")
    axs[i].axis('off')
plt.tight_layout()
plt.show()

# Trích xuất đặc trưng với pretrained model
print("\n2) Trích xuất đặc trưng bằng pretrained model:", model_name)
X_features = extract_features_pretrained(X_raw, model_name=model_name, batch_size=batch_size, pooling=pooling)
print("Features shape:", X_features.shape)

# Lưu đặc trưng
print("\n3) Lưu features và labels ra disk...")
prefix = f"inria_{model_name}_{image_size[0]}"
x_path, y_path = save_features_to_disk(X_features, y_raw, prefix)
print("Saved:", x_path, y_path)
# ...existing code...

NameError: name 'dataset_path' is not defined

In [ ]:
# Import các hàm từ file VS Code của bạn
from modules.image_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk

print("1. Bắt đầu đọc và tiền xử lý ảnh...")
X_raw, y_raw = load_and_preprocess_data(dataset_path)

print("2. Dùng CNN (VGG16/ResNet) để trích xuất đặc trưng...")
X_features = extract_features_pretrained(X_raw)

print("3. Lưu đặc trưng ra file npy...")
# File sẽ được lưu vào thư mục features/ trong repo của bạn [cite: 124]
save_features_to_disk(X_features, y_raw, "inria_cnn")
print("Đã lưu xong!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

print("Đang chia tập dữ liệu Train/Test...")
X_train, X_test, y_train, y_test = train_test_split(X_features, y_raw, test_size=0.2, random_state=42)

print("Đang huấn luyện mô hình SVM...")
model = SVC(kernel='linear')
model.fit(X_train, y_train)

print("Đang đánh giá mô hình trên tập Test...")
y_pred = model.predict(X_test)

# In ra các chỉ số Precision, Recall, F1-Score
print("\nBÁO CÁO KẾT QUẢ PHÂN LOẠI:")
print(classification_report(y_test, y_pred))